# documents

> Document list, detail, and delete routes

In [ ]:
#| default_exp routes.documents

In [ ]:
#| export
from typing import Dict, Callable, Tuple

from fasthtml.common import APIRouter, Div, P

from cjm_transcript_workflow_management.services.management import ManagementService
from cjm_transcript_workflow_management.models import ManagementUrls
from cjm_transcript_workflow_management.components.document_list import render_document_list
from cjm_transcript_workflow_management.components.document_detail import (
    render_document_detail, render_detail_error
)
from cjm_transcript_workflow_management.components.helpers import render_alert
from cjm_transcript_workflow_management.html_ids import ManagementHtmlIds
from cjm_transcript_workflow_management.routes.core import DEBUG_MANAGEMENT_ROUTES

from cjm_fasthtml_daisyui.components.feedback.alert import alert_colors

## Router Initialization

In [ ]:
#| export
def init_document_router(
    service:ManagementService,  # Service for graph queries
    prefix:str,  # Route prefix (e.g., "/manage/documents")
    urls:ManagementUrls,  # URL bundle (populated after init)
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, routes dict)
    """Initialize document list, detail, and delete routes."""
    router = APIRouter(prefix=prefix)
    routes = {}
    
    @router
    async def list_documents(request):
        """Return rendered document list."""
        if DEBUG_MANAGEMENT_ROUTES:
            print("[ROUTES] list_documents called")
        
        documents = await service.list_documents_async()
        
        if DEBUG_MANAGEMENT_ROUTES:
            print(f"[ROUTES] Found {len(documents)} documents")
        
        return render_document_list(documents, urls)
    
    routes["list_documents"] = list_documents
    
    @router
    async def document_detail(request, doc_id:str=""):
        """Return document detail dashboard."""
        if DEBUG_MANAGEMENT_ROUTES:
            print(f"[ROUTES] document_detail called for {doc_id}")
        
        if not doc_id:
            return render_detail_error("No document ID provided.", urls=urls)
        
        detail = await service.get_document_detail_async(doc_id)
        
        if detail is None:
            if DEBUG_MANAGEMENT_ROUTES:
                print(f"[ROUTES] Document not found: {doc_id}")
            return render_detail_error(
                f"Document {doc_id[:12]}... not found.",
                urls=urls
            )
        
        if DEBUG_MANAGEMENT_ROUTES:
            print(f"[ROUTES] Detail loaded: {detail.title}, {detail.segment_count} segments")
        
        return render_document_detail(detail, urls)
    
    routes["document_detail"] = document_detail
    
    @router.post
    async def delete_document(request):
        """Delete a single document and return refreshed list."""
        form_data = await request.form()
        doc_id = form_data.get("doc_id", "")
        
        if DEBUG_MANAGEMENT_ROUTES:
            print(f"[ROUTES] delete_document called for {doc_id}")
        
        if doc_id:
            success = await service.delete_document_async(doc_id)
            if DEBUG_MANAGEMENT_ROUTES:
                print(f"[ROUTES] Delete result: {success}")
        
        # Refresh the document list
        documents = await service.list_documents_async()
        return render_document_list(documents, urls)
    
    routes["delete_document"] = delete_document
    
    @router.post
    async def delete_selected(request):
        """Delete multiple selected documents and return refreshed list."""
        form_data = await request.form()
        doc_ids = form_data.getlist("doc_ids")
        
        if DEBUG_MANAGEMENT_ROUTES:
            print(f"[ROUTES] delete_selected called for {len(doc_ids)} docs")
        
        if doc_ids:
            deleted_count = await service.delete_documents_async(doc_ids)
            if DEBUG_MANAGEMENT_ROUTES:
                print(f"[ROUTES] Deleted {deleted_count} documents")
        
        # Refresh the document list
        documents = await service.list_documents_async()
        return render_document_list(documents, urls)
    
    routes["delete_selected"] = delete_selected
    
    return router, routes

In [ ]:
assert init_document_router is not None
print("routes.documents imports OK")

routes.documents imports OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()